# BTAudit2: inspect, audit, and compare

A read-only tutorial for the DABTROLL research framework. Start with [Getting started](../docs/getting_started.md), the [dataset guide](../docs/btaudit2_dataset.md), and [research attribution](../docs/research_attribution.md).

Install `requirements-analysis.txt` in a Python 3.10+ environment. Obtain the dataset separately; it is not bundled with Git. Launch Jupyter from the repository or this directory. No GPU or mission server is needed for these cells. Outputs are cleared in Git so local reviewer records and stale charts are not published.

The frozen 2026-10-21 snapshot has 488 recordings: 24 episode-1 prompt-tuning recordings and 464 episode-2--21 evaluation recordings. These are different cohorts, not independent prompt-test folds.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'scripts').is_dir() and (p / 'README.md').is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Start Jupyter from the repository or its notebooks directory.')
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
DATASET_ROOT = REPO_ROOT / 'data/logs/btaudit2'  # Change only this if the dataset is elsewhere.
if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(f'Obtain the dataset separately: {DATASET_ROOT}')
COHORT_MANIFEST = REPO_ROOT / 'config/datasets/btaudit2_20261021_manifest.txt'
from audit_btaudit2_dataset import audit_dataset


## 1. Structural audit of the frozen cohort

This reads metadata and workbooks, counts frames, and fingerprints human sheets without saving any input. Inspect `issues` and `warnings` before building statistics. File hashes cover metadata/workbooks, not image pixels. A clean audit establishes structure and provenance, **not visual correctness or complete human annotation**.

In [ ]:
audit = audit_dataset(DATASET_ROOT, manifest=COHORT_MANIFEST)
inventory = pd.DataFrame(audit['episodes'])
display(inventory.groupby(['scenario', 'episode_index']).size().unstack(fill_value=0))
display(inventory[['episode', 'frame_count', 'issues', 'warnings']])
assert audit['episode_count'] == 488, audit['episode_count']
assert audit['issue_episode_count'] == 0, inventory[inventory['issues'].map(bool)]


## 2. Verify the selected full-run prompt

V4 was selected after five episode-1 prompt trials. The following requires one clean matching replay per episode, the saved prompt snapshot/hash, 1024-token budget, and an existing `qwen38_v4` sheet. Failed attempts remain evidence and are not pooled into results. This does not declare that all 464 recordings have completed human review.

In [ ]:
V4_SHA256 = '75adbbaf49c5f7ce77a73229d97d44716e750c48f257f67309dc1805b4c1da50'
v4_audit = audit_dataset(DATASET_ROOT, 2, 21, 'qwen38_v4', V4_SHA256, 1024, COHORT_MANIFEST)
assert v4_audit['episode_count'] == 464
assert v4_audit['issue_episode_count'] == 0
display(pd.DataFrame([{key: v4_audit[key] for key in ('episode_count', 'issue_episode_count', 'clean_responses')}]))


## 3. Episode-1 prompt comparison only

For this reviewed tuning cohort, blank human completion times mean reviewed-incomplete. Missing sheets/nodes are excluded; zero seconds is a valid time. Correct complete = both times present; correct incomplete = neither; false complete = model only; false incomplete = human only. Later node slots require all earlier **model** completions, including false completes. These are presence classifications, not timing accuracy.

V1 has mixed token budgets (22 recordings at 256 and two at 1024), so this is not a strictly matched-budget experiment. Do not extend blank-as-incomplete to an unreviewed full-run cohort. Action-family groupings inferred from node IDs are not simulator ground-truth types.

In [ ]:
from notebook_qwen_compare_utils import build_btaudit2_molmo2_episode1_dataset, build_node_completion_review, plot_node_completion_review
episode1 = build_btaudit2_molmo2_episode1_dataset(DATASET_ROOT, export_csv=False)
assert len(episode1) == 24
comparison = build_node_completion_review(episode1, include_reference_models=False, qwen38_versions=[1, 2, 3, 4, 5])
display(comparison['coverage'])
display(comparison['counts'])
plot_node_completion_review(comparison['counts']);


## 4. Extend without overwriting evidence

Use [the dataset contribution procedure](../docs/btaudit2_dataset.md#adding-recordings-or-human-review) for new recordings/review revisions and [the prompt guide](../docs/status_prompt_experiments.md) for replay. New prompts need new immutable IDs, sheets, and output directories. Keep the published cohort manifest unchanged; create a separately versioned manifest for new cohorts.

Use raw `mission_engine_response` rows for evidence, not synthetic `btstatus_complete_milestone` rows. Keep simulator success, human judgments, raw model status, and effective BT transitions separate. The full-run status replay can stop early after a terminal BT state; absence of a later observation is not proof of task failure. See [notation and limitations](../docs/btaudit2_dataset.md#notation-and-measurement).